# Notebook 01 — Gaussians and Uncertainty

This notebook builds intuition for random variables, Gaussian distributions, and why averaging noisy measurements helps.

## Learning objectives
- Define random variable, mean, and variance in practical terms.
- Understand the 1D Gaussian PDF and how parameters shape uncertainty.
- Compare analytic distributions vs empirical samples.
- Combine multiple noisy measurements of the same quantity.

## Prerequisites
- Python basics and NumPy arrays.
- Basic plotting with Matplotlib.


## Table of Contents

- [Learning objectives](#learning-objectives)
- [Prerequisites](#prerequisites)
- [1) Random variables, mean, variance](#1-random-variables-mean-variance)
- [2) Gaussian PDF intuition](#2-gaussian-pdf-intuition)
- [3) Sampling vs analytic distribution](#3-sampling-vs-analytic-distribution)
- [4) Combining noisy measurements in 1D](#4-combining-noisy-measurements-in-1d)
- [Exercises](#exercises)
- [Common mistakes](#common-mistakes)

---

## Navigation

- Previous: _Start of series_
- Next: [ntbk-02-multivariate-gaussians-covariance.ipynb](./ntbk-02-multivariate-gaussians-covariance.ipynb)

## Relevant source files (repo paths)
- `src/kiss_slam/math_utils.py`


In [ ]:
# Notebook setup (reproducible + matplotlib defaults)
import numpy as np
import matplotlib.pyplot as plt
from _notebook_utils import set_seed

SEED = 100
set_seed(SEED)
plt.rcdefaults()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 7
np.random.seed(SEED)
print(f'Using seed={SEED}')


## 1) Random variables, mean, variance

A random variable $X$ maps outcomes to numbers. In robotics, noisy sensors are modeled as random variables.

- Mean: $\mu = \\mathbb{E}[X]$
- Variance: $\sigma^2 = \\mathbb{E}[(X-\mu)^2]$
- Standard deviation: $\sigma = \\sqrt{\sigma^2}$

For samples $x_1,\dots,x_N$:

$$
\hat\mu = \frac{1}{N}\sum_{i=1}^N x_i,
\qquad
\hat\sigma^2 = \frac{1}{N-1}\sum_{i=1}^N (x_i-\hat\mu)^2.
$$


In [ ]:
true_mu = 5.0
true_sigma = 2.0
samples = np.random.normal(true_mu, true_sigma, size=200)

print('Sample mean:', samples.mean())
print('Sample variance (ddof=1):', samples.var(ddof=1))

plt.figure(figsize=(7,4))
plt.hist(samples, bins=25, density=True, alpha=0.7)
plt.title('Samples from a 1D random variable')
plt.xlabel('x')
plt.ylabel('density')
plt.show()


## 2) Gaussian PDF intuition

A 1D Gaussian distribution is:

$$
p(x) = \frac{1}{\sqrt{2\pi\sigma^2}}\exp\left(-\frac{(x-\mu)^2}{2\sigma^2}\right).
$$

Interpretation:
- $\mu$ shifts the curve.
- $\sigma$ controls spread.


In [ ]:
def gaussian_pdf_1d(x, mu, sigma):
    var = sigma**2
    return (1.0 / np.sqrt(2*np.pi*var)) * np.exp(-0.5 * (x-mu)**2 / var)

x = np.linspace(-5, 15, 500)
for mu in [3.0, 6.0]:
    for sigma in [0.8, 2.5]:
        plt.plot(x, gaussian_pdf_1d(x, mu, sigma), label=f'mu={mu}, sigma={sigma}')
plt.legend()
plt.title('Gaussian PDF parameter sweep')
plt.xlabel('x')
plt.ylabel('p(x)')
plt.show()


## 3) Sampling vs analytic distribution

As sample count grows, the histogram gets closer to the analytic PDF.


In [ ]:
mu, sigma = 2.0, 1.5
x = np.linspace(mu-5*sigma, mu+5*sigma, 600)
pdf = gaussian_pdf_1d(x, mu, sigma)

for N in [30, 200, 5000]:
    s = np.random.normal(mu, sigma, size=N)
    plt.figure(figsize=(7,4))
    plt.hist(s, bins=30, density=True, alpha=0.6, label=f'N={N}')
    plt.plot(x, pdf, 'k', lw=2, label='analytic pdf')
    plt.legend(); plt.show()


## 4) Combining noisy measurements in 1D

If $y_i = z + v_i$ and $v_i\sim\mathcal{N}(0,\sigma^2)$ are independent, then averaging gives:

$$
\mathrm{Var}(\bar{y}) = \frac{\sigma^2}{N}.
$$


In [ ]:
true_value = 10.0
sigma = 3.0
trials = 3000
N_values = [1,2,4,8,16,32]
empirical, analytic = [], []
for N in N_values:
    means = np.array([(true_value + np.random.normal(0, sigma, size=N)).mean() for _ in range(trials)])
    empirical.append(means.var(ddof=1))
    analytic.append(sigma**2 / N)

plt.figure(figsize=(7,4))
plt.plot(N_values, empirical, 'o-', label='empirical')
plt.plot(N_values, analytic, 's--', label='analytic')
plt.xscale('log', base=2); plt.yscale('log')
plt.legend(); plt.title('Variance shrinks with averaging')
plt.show()


## Exercises

1. Estimate mean/variance for N = 20, 200, 2000 and compare error.
2. Change `sigma` and verify variance shrinkage still follows $\sigma^2/N$.


In [ ]:
# Optional solution sketch
for N in [20, 200, 2000]:
    s = np.random.normal(4.0, 1.2, size=N)
    print(N, s.mean(), s.var(ddof=1))


## Common mistakes

- Forgetting `ddof=1` for sample variance.
- Confusing variance with standard deviation.
- Forgetting fixed seeds for reproducibility.


## Exercise Solutions

<details>
<summary>Show solution sketches</summary>

- Re-run the exercise code cells and compare your intermediate values to the reference outputs printed in this notebook.
- Check shapes (`mean`, `covariance`, Jacobians) first; most EKF mistakes are shape/sign issues.
- Use the same `SEED` from the setup cell to keep your run deterministic while debugging.

</details>
